In [ ]:
import os  # استيراد مكتبة os للتعامل مع نظام التشغيل والملفات والمسارات
import shutil  # استيراد مكتبة shutil لعمليات نقل ونسخ الملفات والمجلدات
import random  # استيراد مكتبة random لتوليد أرقام عشوائية وخلط البيانات
import yaml  # استيراد مكتبة yaml لقراءة وإنشاء ملفات التكوين بصيغة YAML
import cv2  # استيراد مكتبة OpenCV (cv2) لمعالجة وتحسين الصور
from ultralytics (
    import YOLO,
)  # استيراد فئة YOLO من مكتبة ultralytics لتدريب واستخدام نماذج YOLO

# 1. تحديد المسارات داخل مجلد الـ fine tune
base_dir = "/content/drive/MyDrive/Old_Russian/finetune"  # تحديد المسار الرئيسي لمجلد المشروع على Google Drive
source_images = os.path.join(
    base_dir, "images"
)  # تحديد مسار مجلد صور المصدر الأصلية
source_labels = os.path.join(
    base_dir, "labels"
)  # تحديد مسار مجلد الملفات النصية للإحداثيات (الليبلز)
output_dir = os.path.join(
    base_dir, "dataset"
)  # تحديد مسار المجلد النهائي المجهز للـ dataset
best_weights_path = os.path.join(
    base_dir, "best.pt"
)  # تحديد مسار ملف أوزان النموذج المدرب سابقاً (best.pt)

# 2. إنشاء الهيكلية الجديدة للـ dataset
for split in ["train", "val"]:  # التكرار على تقسيمات البيانات (التدريب والتحقق)
    os.makedirs(
        os.path.join(output_dir, "images", split), exist_ok=True
    )  # إنشاء مجلد صور التدريب/التحقق إذا لم يكن موجوداً
    os.makedirs(
        os.path.join(output_dir, "labels", split), exist_ok=True
    )  # إنشاء مجلد ليبلز التدريب/التحقق إذا لم يكن موجوداً


# 3. دالة معالجة وتوضيح الصور باستخدام OpenCV مع طباعة تقرير لكل صورة
def enhance_image(
    input_path, output_path
):  # تعريف دالة لمعالجة وتوضيح الصور وتطبيق الفلاتر عليها
    img = cv2.imread(
        input_path
    )  # قراءة الصورة من المسار المحدد باستخدام OpenCV
    if img is None:  # التحقق مما إذا فشلت قراءة الصورة (مثل وجود ملف تالف)
        return False  # إرجاع القيمة False للدلالة على عدم إمكانية معالجة الصورة

    gray = cv2.cvtColor(
        img, cv2.COLOR_BGR2GRAY
    )  # تحويل ألوان الصورة من BGR إلى رمادي (GrayScale) لتسهيل المعالجة
    denoised = cv2.fastNlMeansDenoising(
        gray, h=10
    )  # تطبيق فلتر إزالة الضوضاء والتشويش الرقمي مع تعيين درجة التنعيم h=10
    clahe = cv2.createCLAHE(
        clipLimit=2.0, tileGridSize=(8, 8)
    )  # إنشاء كائن CLAHE لتحسين التباين المحلي بحد تقليم 2.0 وشبكة 8x8
    enhanced = clahe.apply(
        denoised
    )  # تطبيق موازنة التباين التكيفية (CLAHE) على الصورة المنقاة من الضوضاء
    enhanced_rgb = cv2.cvtColor(
        enhanced, cv2.COLOR_GRAY2BGR
    )  # إعادة تحويل الصورة الرمادية المحسنة إلى صيغة BGR لتوافقها مع YOLO

    cv2.imwrite(
        output_path, enhanced_rgb
    )  # حفظ الصورة المعالجة والمحسنة في مسار المخرج الجديد
    return True  # إرجاع القيمة True للدلالة على نجاح عملية المعالجة وحفظ الصورة


# 4. تجميع ونقل البيانات ومعالجتها مع عداد تفصيلي
img_extensions = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
)  # تحديد صيغ واستطالات الصور المقبولة للمشروع
all_images = [
    f
    for f in os.listdir(source_images)
    if f.lower().endswith(img_extensions)
]  # قراءة واستخراج أسماء كافة الصور المطابقة للصيغ المحددة من مجلد الصور

valid_pairs = (
    []
)  # إنشاء قائمة فارغة لتخزين أزواج الصور والليبلز المتطابقة والصالحة
for img_name in all_images:  # المرور بالترتيب على كل اسم صورة في قائمة الصور
    base_name = os.path.splitext(img_name)[
        0
    ]  # فصل اسم الملف عن امتداده للحصول على الاسم الأساسي فقط
    label_name = (
        base_name + ".txt"
    )  # تكوين اسم ملف الليبل المقابل عبر إضافة الامتداد txt.
    lbl_path = os.path.join(
        source_labels, label_name
    )  # إنشاء المسار الكامل لملف الليبل المقابل للصورة

    if os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0:  # التحقق من وجود ملف الليبل المقابل وأن حجمه أكبر من صفر
        valid_pairs.append(
            (img_name, label_name)
        )  # إضافة زوج (اسم الصورة، اسم الليبل) إلى القائمة المعتمدة

random.seed(
    42
)  # تثبيت البذرة العشوائية لضمان تكرار ناتج الخلط بنفس الطريقة دائماً
random.shuffle(
    valid_pairs
)  # خلط قائمة الأزواج المقبولة عشوائياً لضمان توزيع متوازن للبيانات

num_train = int(
    len(valid_pairs) * 0.85
)  # حساب عدد صور التدريب بنسبة 85% من إجمالي الصور الصالحة
train_pairs, val_pairs = (
    valid_pairs[:num_train],
    valid_pairs[num_train:],
)  # تقسيم البيانات إلى قائمة تدريب (85%) وقائمة تحقق (15%)
total_images = len(
    valid_pairs
)  # حساب مجموع الصور الصالحة المعتمدة بالكامل

print(
    f"📊 إجمالي الصور المتطابقة مع الليبلز: {total_images} صورة."
)  # طباعة إجمالي عدد الصور الصالحة والمتطابقة مع الليبلز
print(
    f"🔹 سيتم معالجة وتدريب {len(train_pairs)} صورة، والتحقق من {len(val_pairs)} صورة.\n"
)  # طباعة تفاصيل توزيع البيانات بين مجموعتي التدريب والتحقق


def process_and_copy_with_progress(
    pairs, split
):  # تعريف دالة لمعالجة الصور ونسخ الليبلز وطباعة مؤشر التقدم
    success_count = (
        0  # تهيئة عداد لحساب عدد الصور التي تم معالجتها ونسخها بنجاح
    )
    for idx, (img_name, lbl_name) in enumerate(
        pairs, 1
    ):  # المرور على الأزواج مع استخدام عداد يبدأ من 1
        src_img = os.path.join(
            source_images, img_name
        )  # تحديد المسار الأصلي للصورة المراد معالجتها
        dst_img = os.path.join(
            output_dir, "images", split, img_name
        )  # تحديد المسار الجديد للصورة بعد المعالجة والحفظ

        success = enhance_image(
            src_img, dst_img
        )  # استدعاء دالة معالجة وتوضيح الصورة وحفظ الناتج
        if success:  # في حال نجحت عملية تحسين الصورة وحفظها
            shutil.copy(
                os.path.join(source_labels, lbl_name),
                os.path.join(output_dir, "labels", split, lbl_name),
            )  # نسخ ملف الليبل النصي المقابل للصورة إلى المجلد النهائي
            success_count += (
                1  # زيادة عداد العمليات الناجحة بمقدار 1
            )

        # طباعة حالة التقدم لكل 10 صور أو في النهاية لتشاهد كم عَبَر وكم باقي
        if idx % 10 == 0 or idx == len(
            pairs
        ):  # فحص ما إذا كان التكرار مضاعفاً للعدد 10 أو وصل للصورة الأخيرة
            print(
                f"[{split.upper()}] معالجة الصورة رقم {idx} من {len(pairs)} (تم بنجاح: {success_count})"
            )  # طباعة نص حالة التقدم والعدد الإجمالي المنجز بنجاح


print(
    "⏳ بدء معالجة وتوضيح صور التدريب..."
)  # طباعة رسالة تفيد ببدء مرحلة معالجة وتجهيز صور التدريب
process_and_copy_with_progress(
    train_pairs, "train"
)  # استدعاء دالة المعالجة على مجموعة بيانات التدريب

print(
    "\n⏳ بدء معالجة وتوضيح صور التحقق..."
)  # طباعة رسالة تفيد ببدء مرحلة معالجة وتجهيز صور التحقق
process_and_copy_with_progress(
    val_pairs, "val"
)  # استدعاء دالة المعالجة على مجموعة بيانات التحقق

# 5. استخراج الفئات التلقائي وإنشاء ملف data.yaml
max_class_id = (
    0  # تهيئة متغير لتحديد أعلى رقم فئة (Class ID) موجود في الليبلز
)
for _, lbl_name in (
    valid_pairs
):  # المرور على كافة ملفات الليبلز الصالحة المستخرجة سابقاً
    lbl_file_path = os.path.join(
        source_labels, lbl_name
    )  # تحديد المسار الكامل لملف الليبل النصي الحالي
    with open(
        lbl_file_path, "r", encoding="utf-8"
    ) as f:  # فتح ملف الليبل للقراءة مع ضبط الترميز utf-8
        for line in f:  # قراءة ملف الليبل خطاً بخط (كل خط يمثل كائن محدد)
            parts = (
                line.strip().split()
            )  # تقطيع السطر إلى أجزاء بحسب الفواصل الخالية
            if parts:  # التحقق من أن السطر ليس فارغاً ويحتوي بيانات
                class_id = int(
                    parts[0]
                )  # تحويل العنصر الأول من السطر إلى عدد صحيح وهو معرّف الفئة
                if class_id > max_class_id:  # فحص ما إذا كان معرّف الفئة الحالي أكبر من القيمة القصوى السابقة
                    max_class_id = (
                        class_id  # تحديث أعلى رقم فئة تم العثور عليه حتى الآن
                    )

num_classes = (
    max_class_id + 1
)  # حساب عدد الفئات الكلي بزيادة 1 على أعلى رقم فئة (لأن الترقيم يبدأ من 0)
names_dict = {
    i: f"class_{i}" for i in range(num_classes)
}  # إنشاء قاموس يربط كل معرّف فئة باسم افتراضي class_i

yaml_data = {  # إنشاء قاموس يحتوي على هيكلية وشروط ملف التكوين YAML
    "path": output_dir,  # تحديد المسار الرئيسي لمجلد البيانات الشامل
    "train": "images/train",  # تحديد المسار النسبي لمجلد صور التدريب داخل path
    "val": "images/val",  # تحديد المسار النسبي لمجلد صور التحقق داخل path
    "names": names_dict,  # إضافة قاموس أسماء الفئات إلى ملف التكوين
}  # إغلاق تعريف القاموس yaml_data

yaml_path = os.path.join(
    output_dir, "data.yaml"
)  # تحديد المسار النهائي لحفظ ملف data.yaml
with open(
    yaml_path, "w", encoding="utf-8"
) as f:  # فتح ملف data.yaml للكتابة مع ضبط التشفير utf-8
    yaml.dump(
        yaml_data, f, default_flow_style=False, allow_unicode=True
    )  # كتابة بيانات القاموس داخل ملف YAML مع دعم الأحرف والتنسيق القابل للقراءة

print(
    f"\n✅ تمت المعالجة وتوليد ملف data.yaml لـ {num_classes} فئة بنجاح.\n"
)  # طباعة تأكيد نجاح توليد ملف YAML وتوضيح عدد الفئات الكلي

# 6. تحميل النموذج وبدء الـ Fine-tuning
print(
    "🚀 بدء تدريب النموذج (Fine-tuning) عبر YOLO..."
)  # طباعة رسالة تفيد ببدء عملية إعادة ضبط وتدريب النموذج
model = YOLO(
    best_weights_path
)  # تحميل نموذج YOLO باستخدام ملف الأوزان السابقة (best.pt)

results = model.train(  # استدعاء دالة التدريب وتمرير معلمات الإعدادات للنموذج
    data=yaml_path,  # تمرير مسار ملف التكوين data.yaml للنموذج
    epochs=120,  # تحديد عدد دورات التدريب الكلية (120 epoch)
    imgsz=1040,  # ضبط أبعاد حجم الصورة المدخلة للنموذج أثناء التدريب بـ 1040 بكسل
    batch=4,  # تحديد حجم الدفعة الواحدة (Batch size) ليكون 4 صور
    lr0=0.0001,  # ضبط معدل التعلم الأولي (Initial Learning Rate) بـ 0.0001
    freeze=10,  # تجميد أول 10 طبقات من النموذج للحفاظ على الميزات المستخرجة سابقاً
    mosaic=0.1,  # ضبط معامل تجميع الصور (Mosaic Augmentation) بنسبة 0.1
    box=7.5,  # تحديد وزن معامل خسارة صناديق التحديد (Box loss weight) بـ 7.5
    patience=50,  # ضبط التوقف المبكر عند عدم تحسن النموذج بعد 50 دورة متتالية
    project=base_dir,  # تحديد المجلد الرئيسي الذي سيتم حفظ نتائج التدريب بداخله
    name="finetune_enhanced_results",  # تحديد اسم مجلد النتائج الخاص بهذه الجلسة التدريبية
)  # إغلاق قوس دالة التدريب model.train

In [ ]:
import json  # استيراد مكتبة json لقراءة والتعامل مع ملفات التكوين وقواميس البيانات بصيغة JSON
import os  # استيراد مكتبة os للتحقق من وجود الملفات والتعامل مع مسارات نظام التشغيل
import traceback  # استيراد مكتبة traceback لتتبع تفاصيل الأخطاء والاحتفاظ بتقرير الخطأ الكامل
import numpy as np  # استيراد مكتبة numpy للعمليات الحسابية والمعالجة الرياضية للمصفوفات والإحداثيات
import gradio as gr  # استيراد مكتبة gradio لبناء وتصميم واجهات المستخدم التفاعلية لتطبيقات الذكاء الاصطناعي
from pathlib import Path  # استيراد كلاس Path من مكتبة pathlib لإنشاء والتعامل مع مسارات الملفات بشكل كائني
from PIL import Image, ImageDraw  # استيراد أدوات PIL لقراءة الصور المرفوعة والرسم عليها (مثل رسم صناديق التحديد)
from ultralytics import YOLO  # استيراد فئة YOLO من مكتبة ultralytics لتحميل نموذج كشف الحروف وإجراء التوقعات

# مسار نموذج الـ Fine-Tuning وملف الـ alphabet.json
FINAL_MODEL = Path("/content/drive/MyDrive/Old_Russian/finetune/finetune_enhanced_results/weights/best.pt")  # تحديد المسار المطلق لملف أوزان النموذج النهائي المتدرب
ALPHABET_FILE = Path("/content/drive/MyDrive/Old_Russian/finetune/Alphabet.json")  # تحديد مسار ملف الأبجدية JSON المحتوي على قاموس ترميز الحروف

def load_inference_model():  # تعريف دالة مخصصة لتحميل نموذج YOLO المستهدف للاستنتاج
    model_path = str(FINAL_MODEL)  # تحويل كائن المسار Path إلى نص String ليتوافق مع مكتبة YOLO
    if not os.path.exists(model_path):  # التحقق من عدم وجود ملف النموذج في المسار المحدد
        raise FileNotFoundError(f"لم يتم العثور على ملف النموذج في المسار: {model_path}")  # إطلاق استثناء خطأ في حال عدم العثور على ملف النموذج
    return YOLO(model_path)  # تحميل النموذج وإعادة كائن YOLO الجاهز للتوقع

# تحميل قاموس الحروف بدقة متناهية من ملف alphabet.json المعكوس بناءً على char_to_id
ID_TO_CHAR = {}  # إنشاء قاموس فارغ لتخزين الربط بين رقم الفئة (ID) والحرف السلافي المناظر له
if ALPHABET_FILE.exists():  # التحقق مما إذا كان ملف الأبجدية JSON موجوداً بالفعل في المسار
    try:  # بدء كتلة محاولة قراءة ملف JSON وتفادي الأخطاء
        with open(ALPHABET_FILE, 'r', encoding='utf-8') as f:  # فتح ملف JSON للقراءة بترميز utf-8 لدعم الحروف الخاصة
            data = json.load(f)  # تحميل وتحويل محتوى ملف JSON إلى قاموس بيانات بايثون
            # عكس القاموس لربط الـ ID (القيمة) بالحرف الصحيح (المفتاح)
            char_to_id = data.get("char_to_id", {})  # جلب قاموس char_to_id من البيانات المحملة أو قاموس فارغ افتراضياً
            ID_TO_CHAR = {v: k for k, v in char_to_id.items()}  # عكس القاموس لتصبح القيم (IDs) هي المفاتيح والحروف هي القيم
        print(f"✅ تم تحميل وترتيب الحروف بنجاح من ملف alphabet.json! (عدد الحروف: {len(ID_TO_CHAR)})")  # طباعة رسالة تأكيد نجاح تحميل القاموس مع عدد الحروف
    except Exception as e:  # التقاط أي استثناء قد يحدث أثناء قراءة أو معالجة ملف JSON
        print(f"⚠️ حدث خطأ أثناء قراءة ملف alphabet.json: {e}")  # طباعة رسالة تحذير مع تفاصيل الخطأ الذي حدث

# في حال لم يتم تحميل الملف، يتم الاعتماد على أسماء النموذج الافتراضية كاحتياط
if not ID_TO_CHAR:  # فحص ما إذا كان قاموس الحروف لا يزال فارغاً (بسبب عدم وجود الملف أو حدوث خطأ)
    try:  # بدء كتلة محاولة جلب الأسماء الافتراضية من النموذج نفسه
        temp_model = load_inference_model()  # تحميل نسخة مؤقتة من النموذج لاستخراج الأسماء المدمجة به
        ID_TO_CHAR = temp_model.names  # تعيين أسماء الفئات المدمجة بالنموذج إلى القاموس الاحتياطي
        print("⚠️ تم الاعتماد على أسماء النموذج الداخلية.")  # طباعة رسالة تحذير بأنه تم استخدام الأسماء الافتراضية للنموذج
    except:  # التقاط أي خطأ يحدث أثناء تحميل النموذج الاحتياطي
        pass  # تجاهل الخطأ في حال الاستثناء ومواصلة تنفيذ البرنامج


def group_boxes_into_text(boxes_xyxy, cls_ids, line_thresh_ratio=0.6, space_factor=0.5):  # تعريف دالة ترتيب وتجميع الصناديق لتكوين أسطر وكلمات النص
    if len(boxes_xyxy) == 0:  # فحص ما إذا كانت قائمة الصناديق المكتشفة فارغة
        return ""  # إعادة نص فارغ مباشرة في حالة عدم وجود أي صناديق مكتشفة

    heights = boxes_xyxy[:, 3] - boxes_xyxy[:, 1]  # حساب ارتفاع كل صندوق كشف عبر طرح y1 من y2
    median_h = np.median(heights) if len(heights) else 20  # حساب الوسيط الإحصائي لارتفاع الحروف أو فرض 20 كقيمة افتراضية
    y_centers = (boxes_xyxy[:, 1] + boxes_xyxy[:, 3]) / 2  # حساب الإحداثي العمودي لمراكز الصناديق (y1 + y2) / 2
    order = np.argsort(y_centers)  # الحصول على الترتيب التصاعدي لمؤشرات الصناديق بناءً على مراكزها العمودية

    lines = []  # إنشاء قائمة فارغة لتخزين مجاميع الصناديق المقسمة حسب الأسطر
    current = [order[0]]  # بدء السطر الأول بأول صندوق محدد في الترتيب العمودي
    current_y = y_centers[order[0]]  # حفظ مركز y للصندوق الأول ليكون المرجع للسطر الحالي

    for idx in order[1:]:  # التكرار على بقية مؤشرات الصناديق المرتبة عمودياً
        if abs(y_centers[idx] - current_y) <= median_h * line_thresh_ratio:  # التحقق من وقوع الحرف على نفس السطر بناءً على المسافة العمودية وعتبة السطر
            current.append(idx)  # إضافة مؤشر الحرف الحالي إلى القائمة الخاصة بالسطر الحالي
        else:  # في حال تجاوز المسافة العمودية العتبة المحددة (بداية سطر جديد)
            lines.append(current)  # حفظ قائمة حروف السطر السابق في قائمة الأسطر الكلية
            current = [idx]  # إنشاء سطر جديد يحتوي على مؤشر الحرف الحالي
            current_y = y_centers[idx]  # تحديث مركز y المرجعي ليعبر عن السطر الجديد
    lines.append(current)  # إضافة السطر الأخير إلى قائمة الأسطر بعد انتهاء الحلقة التكرارية

    output = []  # إنشاء قائمة فارغة لتخزين النصوص النهائية لكل سطر على حدة
    for line in lines:  # المرور على كل سطر تم تجميعه لمعالجة ترتيب حروفه وأبعاده
        line = sorted(line, key=lambda i: boxes_xyxy[i][0])  # إعادة ترتيب الحروف داخل السطر الواحد أفقياً من اليسار إلى اليمين بناءً على x1
        line_chars = []  # قائمة فارغة لتجميع أحرف وفواصل السطر الحالي
        widths = [boxes_xyxy[i][2] - boxes_xyxy[i][0] for i in line]  # حساب عرض كل صندوق حرف في السطر الحالي (x2 - x1)
        avg_w = np.mean(widths) if widths else 15  # حساب متوسط عرض الحروف في السطر الحالي أو فرض 15 كعرض افتراضي

        for i, idx in enumerate(line):  # المرور بالتكرار والترتيب على الحروف داخل السطر الحالي
            if i > 0:  # تطبيق فحص الفجوات الأفقية بين الحروف بداية من الحرف الثاني
                prev_x2 = boxes_xyxy[line[i-1]][2]  # استخراج الإحداثي الأفقي الأيمن (x2) للحرف السابق
                curr_x1 = boxes_xyxy[idx][0]  # استخراج الإحداثي الأفقي الأيسر (x1) للحرف الحالي
                gap = curr_x1 - prev_x2  # حساب الفجوة الأفقية الفاصلة بين الحرف السابق والحالي
                if gap > (avg_w * space_factor):  # فحص ما إذا كانت الفجوة تتجاوز العتبة المحددة للمسافة بين الكلمات
                    line_chars.append(" ")  # إضافة مسافة خالية بين الكلمات عند تحقق الشرط

            char = ID_TO_CHAR.get(int(cls_ids[idx]), f"?[{int(cls_ids[idx])}]")  # استخراج الحرف المترجم من القاموس أو إدراج علامة مجهول مع رقم الفئة
            line_chars.append(char)  # إلحاق الحرف بقائمة أحرف السطر الحالي

        output.append("".join(line_chars))  # دمج كافة أحرف وفواصل السطر في سلسلة نصية واحدة وإضافتها للنتائج

    return "\n".join(output)  # دمج أسطر النص بفاصل السطر الجديد (\n) وإرجاع النص الكامل


def draw_bboxes(image, boxes_xyxy, cls_ids, confs):  # تعريف دالة לרسم الصناديق والوسوم الإرشادية على الصورة
    image = image.convert("RGB").copy()  # تحويل نظام ألوان الصورة إلى RGB وأخذ نسخة منها للرسم والتعديل
    draw = ImageDraw.Draw(image)  # إنشاء كائن أداة الرسم فوق الصورة
    for box, cid, conf in zip(boxes_xyxy, cls_ids, confs):  # التكرار على الصناديق ومعرفات الفئات ودرجات الثقة بالتوازي
        x1, y1, x2, y2 = map(int, box)  # تحويل إحداثيات الصندوق إلى أعداد صحيحة لتحديد بكسلات الرسم
        draw.rectangle([x1, y1, x2, y2], outline="red", width=2)  # رسم مستطيل تحديد باللون الأحمر وبسمك 2 بكسل حول الحرف
        char_name = ID_TO_CHAR.get(int(cid), f"?({cid})")  # الحصول على اسم الحرف المناظر لرقم الفئة أو إظهار رمز مجهول
        label = f"{char_name} ({conf:.2f})"  # صياغة نص الوسم الذي يضم اسم الحرف مع نسبة ثقة الكشف
        draw.text((x1, max(0, y1 - 15)), label, fill="blue")  # كتابة نص الوسم باللون الأزرق أعلى المستطيل
    return image  # إعادة كائن الصورة المعدلة بعد رسم الصناديق والوسوم عليها


_INFERENCE_MODEL = None  # تهيئة متغير عام بقيمة None لتخزين كائن النموذج وتجنب إعادة تحميله مع كل طلب


def process_image(image, conf_thresh, line_ratio):  # تعريف دالة معالجة واستنتاج الصورة الرئيسية المرتبطة بزر الواجهة
    global _INFERENCE_MODEL  # الإشارة إلى استخدام المتغير العام الخاص بنموذج الاستنتاج
    try:  # بدء كتلة المعالجة الآمنة لالتقاط أي خطأ أثناء الاستنتاج
        if image is None:  # التحقق مما إذا كان المستخدم لم يقم برفع صورة في الواجهة
            return None, "الرجاء رفع صورة أولاً."  # إعادة قيمة فارغة مع رسالة إرشادات للمستخدم

        if _INFERENCE_MODEL is None:  # التحقق مما إذا كان النموذج لم يتم تحميله في الذاكرة بعد
            _INFERENCE_MODEL = load_inference_model()  # تحميل النموذج واستبقائه داخل المتغير العام

        results = _INFERENCE_MODEL.predict(source=image, conf=conf_thresh, imgsz=1056, verbose=False)  # إجراء التوقع وتحديد الحروف بحد ثقة ومقاس صورة 1056

        if len(results) == 0 or results[0].boxes is None or len(results[0].boxes) == 0:  # فحص عدم وجود أي ناتج كشف أو عدم استخراج صناديق
            return image, "لم يتم العثور على أي حروف مطابقة بالثقة المحددة."  # إعادة الصورة الأصلية مع رسالة عدم العثور على كشوفات

        boxes = results[0].boxes.xyxy.cpu().numpy()  # استخراج إحداثيات الصناديق وتحويلها إلى مصفوفة numpy
        cls_ids = results[0].boxes.cls.cpu().numpy()  # استخراج أرقام الفئات المكتشفة وتحويلها لمصفوفة numpy
        confs = results[0].boxes.conf.cpu().numpy()  # استخراج درجات الثقة لكل كشف وتحويلها لمصفوفة numpy

        annotated = draw_bboxes(image, boxes, cls_ids, confs)  # رسم صناديق التحديد والوسوم على الصورة المرفوعة
        text = group_boxes_into_text(boxes, cls_ids, line_thresh_ratio=line_ratio)  # استخراج النص وترتيب الحروف في أسطر وكلمات

        return annotated, text  # إعادة الصورة المعلمة بمرسمات الكشف والنص المترجم النهائي

    except Exception:  # التقاط كافة الاستثناءات والأخطاء أثناء معالجة الصورة
        err = traceback.format_exc()  # استخراج التقرير التفصيلي للخطأ الحاصل
        print(err)  # طباعة تفاصيل الخطأ في شاشة العرض البرمجية
        return image, err  # إعادة الصورة الأصلية متبوعة بنص التقرير المفصل للخطأ للواجهة


# تصميم واجهة Gradio الكاملة
with gr.Blocks(title="OCR — المخطوطات السلافية القديمة") as demo:  # إنشاء وتصميم واجهة Gradio باستخدام نظام الكتل Blocks
    gr.Markdown("# 📜 نظام التعرف البصري على الحروف (OCR) - المخطوطات السلافية القديمة")  # إضافة عنوان رئيسي ملون بدعم تنسيق ماركداون
    gr.Markdown("قم برفع صورة المخطوط الحقيقي، واضبط إعدادات الكشف، ثم اضغط على **استخراج النص**.")  # إضافة وصف تعليمي للمستخدم بكيفية الاستخدام

    with gr.Row():  # إنشاء صف تخطيط أفقي لتقسيم العناصر إلى أعمدة متوازية
        with gr.Column(scale=1):  # إنشاء العمود الأول (جانب الإدخال والتحكم) بعرض نسبي 1
            img_in = gr.Image(type="pil", label="ارفع صورة المخطوط الحقيقي")  # عنصر إدخال الصورة وإعادتها بصيغة PIL

            with gr.Accordion("إعدادات متقدمة", open=False):  # قائمة منسدلة مغلقة افتراضياً للإعدادات المتقدمة
                conf_slider = gr.Slider(minimum=0.05, maximum=0.9, value=0.15, step=0.05, label="عتبة الثقة (Confidence Threshold)")  # شريط التحكم بعتبة درجة ثقة النموذج
                line_slider = gr.Slider(minimum=0.2, maximum=1.5, value=0.6, step=0.1, label="حساسية تجميع الأسطر (Line Spacing Ratio)")  # شريط التحكم بحساسية تقطيع وتجميع الأسطر

            btn = gr.Button("🔍 استخراج النص وتحديد الحروف", variant="primary")  # زر تنفيذ عملية الكشف بتصميم مميز

        with gr.Column(scale=1):  # إنشاء العمود الثاني (جانب المخرجات والنتائج) بعرض نسبي 1
            img_out = gr.Image(label="الصورة مع مربعات التحديد (Bounding Boxes)")  # عنصر عرض الصورة الناتجة بعد الرسم عليها
            txt_out = gr.Textbox(label="النص المستخرج", lines=10, max_lines=20)  # عنصر عرض النص المستخرج بداخل صندوق نصي قابل للتمرير

    btn.click(  # إعداد حدث الضغط على زر الاستخراج
        fn=process_image,  # ربط زر الضغط بدالة المعالجة والاستنتاج الرئيسية
        inputs=[img_in, conf_slider, line_slider],  # إسناد المدخلات الممررة إلى الدالة (الصورة، عتبة الثقة، حساسية الأسطر)
        outputs=[img_out, txt_out]  # إسناد العناصر المستلمة لمخرجات الدالة (الصورة المعلمة، النص)
    )

demo.queue()  # تفعيل نظام قائمة الانتظار لتنظيم طلبات الاستنتاج المتعددة
demo.launch(share=True, debug=True)  # تشغيل خادم التطبيق وإتاحة رابط مشاركة خارجي مع تفعيل وضع التنقيح

✅ تم تحميل قاموس الحروف بنجاح من ملف classes.txt! (عدد الحروف: 38)
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8d323679bc11fd2b5a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://1a817f7a2b1b239e3a.gradio.live
Killing tunnel 127.0.0.1:7860 <> https://8d323679bc11fd2b5a.gradio.live
